In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/marcos.moretti/repos/conformal-factual-lm/venv-cf-2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "meta-llama/Llama-3.1-8B"

In [3]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [13]:
# Load model with safe dtype + multi-GPU mapping
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,          # safer than bfloat16 if you see errors
    device_map="auto",            # automatically splits across GPUs
    offload_folder="offload",     # optional: spill to CPU if needed
)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2365.42it/s, Materializing param=model.norm.weight]                              


In [14]:
model.device

device(type='cpu')

In [15]:
# Prepare input
prompt = "Hello World!"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # pt = pytorch
#inputs = tokenizer(prompt, return_tensors="pt")
#inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.7
)

# Decode
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Hello World! My name is M. S. F. M. I am a 25 year old girl. I have been an avid reader since childhood. My passion for reading led me to become an English teacher. I taught for 4 years in a secondary school in my country. After that, I decided to follow my dream and study English literature in a university. I graduated from the University of Dhaka in 2015. Currently, I am working as a lecturer at a university. I love to


In [ ]:
import torch
print(torch.cuda.device_count())   # Should be 4
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

In [ ]:
import torch
print(model.device)  # what device is the model on?
print(torch.cuda.is_available())  # should be True


In [10]:
inputs = tokenizer(prompt, return_tensors="pt")
print(inputs["input_ids"])

tensor([[128000,   9906,   4435,      0]])


In [12]:
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
